In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import matplotlib.pyplot as plt
from src.data.synthetic import generate_dataset
from src.ner.train import finetune, _align_labels
from src.ner.evaluate import compute_metrics
from src.ner.model import load_finetuned, extract_entities

In [ ]:
all_docs = generate_dataset(n=1500, seed=42)
split = int(len(all_docs) * 0.85)
train_docs, val_docs = all_docs[:split], all_docs[split:]
print(f"NER Train: {len(train_docs)}, Val: {len(val_docs)}")

In [ ]:
# Fine-tune BERT-NER (~30-45 min on Kaggle T4 GPU)
model_dir = finetune(train_docs, val_docs)
print(f"NER model saved to: {model_dir}")

In [ ]:
model, tokenizer, id2label = load_finetuned(model_dir)

all_preds, all_refs = [], []
for doc in val_docs[:100]:
    entities = extract_entities(doc.text, model, tokenizer, id2label)
    pred_labels = ["O"] * len(doc.text.split())
    for e in entities:
        words = e.value.split()
        doc_words = doc.text.split()
        for i in range(len(doc_words) - len(words) + 1):
            if doc_words[i:i+len(words)] == words:
                pred_labels[i] = f"B-{e.label}"
                for j in range(1, len(words)):
                    pred_labels[i + j] = f"I-{e.label}"
                break
    all_preds.append(pred_labels)
    all_refs.append(_align_labels(doc.text.split(), doc))

metrics = compute_metrics(all_preds, all_refs)
print(f"Overall  P: {metrics['precision']:.3f}  R: {metrics['recall']:.3f}  F1: {metrics['f1']:.3f}")

In [ ]:
rows = []
for label, scores in metrics["per_label"].items():
    rows.append({
        "Label": label,
        "Precision": round(scores["precision"], 3),
        "Recall": round(scores["recall"], 3),
        "F1": round(scores["f1-score"], 3),
        "Support": scores["number"],
    })
df = pd.DataFrame(rows).set_index("Label")
print(df.to_string())